# Copy-move forgeries + masks (issues #5, #24)

Takes the authentic patches made by the tiling notebook, makes one forged twin
per patch with its mask, writes the manifest, and puts the dataset on Drive as one zip.
Runs on a CPU runtime (no GPU needed).

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | Copy the authentic patches from Drive to the VM |
| 4 | Generate the forgeries and the manifest |
| 5 | Check the dataset: counts and paste sizes |
| 6 | Look at a few forgeries |
| 7 | Zip the dataset onto Drive |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

PATCHES_ZIP = DRIVE_ROOT + "/data/patches_v0.1.zip"
LEVEL = 1                                          # difficulty level (#24-#27)
DATASET_ZIP = DRIVE_ROOT + f"/data/dataset_v0.1_level{LEVEL}.zip"
LOCAL_DIR = "/content/dataset"
SEED = 0

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

# The notebook runs from its own folder and imports the module next to it.
os.chdir(REPO_DIR + "/features/copy_move")
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
from features.sar_io.sar_io import unzip_folder

unzip_folder(PATCHES_ZIP, LOCAL_DIR)

In [ ]:
from copy_move import build_dataset

manifest = build_dataset(LOCAL_DIR, level=LEVEL, seed=SEED)

In [ ]:
print(manifest.groupby(["split", "label"]).size())
forged = manifest[manifest["label"] == 1]
print("Paste size, % of the patch:")
print((100 * forged["paste_fraction"]).describe().round(2))

In [ ]:
import matplotlib.pyplot as plt

from copy_move import show_forgery

for patch_id in forged.sample(3, random_state=SEED)["patch_id"]:
    show_forgery(LOCAL_DIR, patch_id)
    plt.show()

In [ ]:
from features.sar_io.sar_io import zip_folder

zip_folder(LOCAL_DIR, DATASET_ZIP)